In [2]:
import pandas as pd
import numpy as np

# Leer el archivo limpio
df = pd.read_csv("alerce_data/all_lightcurves_clean.csv")

# Verificamos que las columnas necesarias están presentes
required_columns = {"oid", "magpsf", "sigmapsf"}
if not required_columns.issubset(df.columns):
    raise ValueError(f"Faltan columnas necesarias: {required_columns - set(df.columns)}")

# Definir función para estadísticas por OID
def compute_statistics(group):
    mags = group["magpsf"].values
    errs = group["sigmapsf"].values

    variance = np.var(mags, ddof=1)
    mean_err_sq = np.mean(errs**2) if not np.isnan(errs).all() else 0
    excess_var = variance - mean_err_sq
    excess_var = excess_var if excess_var > 0 else 0

    return pd.Series({
        "n_points": len(mags),
        "mag_min": np.min(mags),
        "mag_max": np.max(mags),
        "mag_p05": np.percentile(mags, 5),
        "mag_p25": np.percentile(mags, 25),
        "mag_p50": np.percentile(mags, 50),
        "mag_p75": np.percentile(mags, 75),
        "mag_p95": np.percentile(mags, 95),
        "excess_variance": excess_var
    })

# Agrupar y aplicar
stats_df = df.groupby("oid").apply(compute_statistics).reset_index()

# Guardar en CSV
output_path = "alerce_data/statistics_lightcurve_alerts.csv"
stats_df.to_csv(output_path, index=False)

print(f"✅ Archivo guardado como: {output_path}")
print(stats_df.head())


c:\Users\adegr\anaconda3\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:3787: RuntimeWarning: Degrees of freedom <= 0 for slice
  return _methods._var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\adegr\anaconda3\anaconda3\Lib\site-packages\numpy\core\_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


✅ Archivo guardado como: alerce_data/statistics_lightcurve_alerts.csv
            oid  n_points    mag_min    mag_max    mag_p05    mag_p25  \
0  ZTF17aaaeqms      98.0  17.583920  19.228844  17.832155  17.920535   
1  ZTF17aaaeucx      22.0  19.444640  24.400576  20.032027  20.481994   
2  ZTF17aaakbyl       6.0  18.844500  21.830654  19.127626  20.057251   
3  ZTF17aaakyau      19.0  19.580053  22.262995  19.706568  19.891437   
4  ZTF17aaapmss      11.0  18.673641  19.389822  18.740383  18.834072   

     mag_p50    mag_p75    mag_p95  excess_variance  
0  18.011668  18.225811  18.565368         0.058497  
1  21.447725  22.856154  23.800645         0.000000  
2  20.816783  21.379518  21.721532         0.465225  
3  20.669268  21.033121  21.559356         0.000000  
4  18.887829  18.925891  19.189695         0.020169  
